# Train SmolVLA on the garage dataset  (Google Colab, T4)  - dataset from Hugging Face, no Google Drive needed

**On your laptop first (once):**
1. Create a free account at huggingface.co and a token with **write** access (Settings -> Access Tokens).
2. `pip install -U huggingface_hub` then `huggingface-cli login`
3. `python upload_dataset_hf.py --repo <your-username>/garage_pnp`  (uploads `demo_data_garage` as a **private** dataset; resumable - if it stops, run it again)

**On Colab:** Runtime -> Change runtime type -> **T4 GPU**. In the key icon on the left add a secret named `HF_TOKEN` (or you will be asked to paste the token).
Then run the cells top to bottom. Checkpoints are saved to your private Hugging Face model repo (`CKPT_STORE = "hf"`) so a disconnect only costs the time since the last save.

**Sizing for ~1000 scripted episodes (about 150k frames):** one pass over the data is 150k / batch 8 = 18,750 steps, about 8.7 h on a T4.
`STEPS = 20_000` is therefore about one pass (about 9.3 h, i.e. two or three Colab sessions, resume between them). More *distinct episodes* help more than extra passes over the same ones, so we train on all of them.

In [ ]:
!nvidia-smi

### Install
If this cell makes Colab ask to restart, do **Runtime -> Restart session** and continue from the *Settings* cell (do not re-run the install).

In [ ]:
# same lerobot commit the tutorial repo pins (newer lerobot moved lerobot.common and breaks train_model.py)
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git@10b7b3532543b4adfb65760f02a49b4c537afde7"
!pip install -q transformers==4.50.3 num2words accelerate safetensors==0.5.3 datasets==3.4.1

In [ ]:
import os, re, json, shutil, subprocess, threading, time
from pathlib import Path

# ----------------------------- SETTINGS -----------------------------
STEPS       = 20_000      # ~1 pass over 1000 episodes at batch 8 (~9.3 h on a T4). 12_000 if you are short on time.
BATCH_SIZE  = 8           # a T4 cannot hold more (2 images per sample at 512x512)
CHUNK       = 10          # chunk_size = n_action_steps. Deployment reads it from the checkpoint, nothing to change there.
SAVE_FREQ   = 2_500       # a restart point every ~70 minutes
MAX_EPISODES = None       # None = all episodes. e.g. 500 = use (and download) only the first 500

HF_DATASET_REPO = None    # None -> "<your-username>/garage_pnp"
HF_MODEL_REPO   = None    # None -> "<your-username>/garage-smolvla-ckpt"  (private, created automatically)
CKPT_STORE  = "hf"        # "hf" (recommended, no Drive needed) or "drive"
DRIVE_DIR   = "/content/drive/MyDrive/garage"      # only used when CKPT_STORE == "drive"

REPO_NAME   = "garage_pnp"                         # the dataset's own repo_id (as recorded)
WORK        = "/content/work"
DATA_ROOT   = "/content/demo_data_garage"
LOCAL_OUT   = "/content/ckpt/smolvla_garage"
# --------------------------------------------------------------------
os.makedirs(WORK, exist_ok=True)

### Hugging Face login

In [ ]:
from huggingface_hub import HfApi, login, snapshot_download
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    token = getpass.getpass("Paste your Hugging Face token (write access): ")
login(token=token)
api = HfApi()
HF_USER = api.whoami()["name"]
HF_DATASET_REPO = HF_DATASET_REPO or f"{HF_USER}/garage_pnp"
HF_MODEL_REPO   = HF_MODEL_REPO   or f"{HF_USER}/garage-smolvla-ckpt"
print("logged in as", HF_USER, "| dataset:", HF_DATASET_REPO, "| checkpoints:", HF_MODEL_REPO if CKPT_STORE == "hf" else DRIVE_DIR)

if CKPT_STORE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")

### Download the dataset (about 18 GB for 1000 episodes; takes minutes on Colab)
The first training start afterwards also builds a local cache from these files (another few minutes, similar extra disk). Colab has about 78 GB, enough.

In [ ]:
allow = None
if MAX_EPISODES is not None:
    allow = ["meta/*"] + [f"data/*/episode_{i:06d}.parquet" for i in range(MAX_EPISODES)]
snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_ROOT, allow_patterns=allow, max_workers=8)
assert os.path.exists(f"{DATA_ROOT}/meta/info.json"), "download failed: meta/info.json is missing"
!du -sh {DATA_ROOT}
!df -h /content | tail -1

# training script (identical to the one in the tutorial repo)
!wget -q -O {WORK}/train_model.py https://raw.githubusercontent.com/jeongeun980906/lerobot-mujoco-tutorial/master/train_model.py

In [ ]:
# sanity-check the dataset before spending GPU time
info = json.load(open(f"{DATA_ROOT}/meta/info.json"))
n_eps = info["total_episodes"] if MAX_EPISODES is None else min(MAX_EPISODES, info["total_episodes"])
EPISODES = None if MAX_EPISODES is None else list(range(n_eps))
frames = info["total_frames"] if MAX_EPISODES is None else int(info["total_frames"] * n_eps / info["total_episodes"])
print("episodes used:", n_eps, "| about", frames, "frames | fps", info["fps"])
for k, v in info["features"].items():
    print(f"  {k:28s} {v['dtype']:8s} {v['shape']}")
print("tasks:")
for line in open(f"{DATA_ROOT}/meta/tasks.jsonl"):
    print("  ", json.loads(line))
assert info["features"]["observation.state"]["shape"] == [8] and info["features"]["action"]["shape"] == [8]
if EPISODES is not None:
    missing = [i for i in EPISODES if not list(Path(DATA_ROOT).glob(f"data/*/episode_{i:06d}.parquet"))]
    assert not missing, f"episode files missing: {missing[:5]}..."
print(f"\n{STEPS} steps x batch {BATCH_SIZE} = {STEPS*BATCH_SIZE} samples = {STEPS*BATCH_SIZE/frames:.2f} passes over the data;"
      f" about {STEPS*1.67/3600:.1f} h on a T4 at 1.67 s/step")

In [ ]:
episodes_line = f"\n  episodes: {json.dumps(EPISODES)}" if EPISODES is not None else ""
yaml_text = f'''
dataset:
  repo_id: {REPO_NAME}
  root: {DATA_ROOT}{episodes_line}
policy:
  type: smolvla
  chunk_size: {CHUNK}
  n_action_steps: {CHUNK}
  device: cuda
  use_amp: false            # T4 has no bf16
  scheduler_decay_steps: {STEPS}   # the learning rate finishes decaying at the last step (default is 30000)
output_dir: {LOCAL_OUT}
batch_size: {BATCH_SIZE}
job_name: smolvla_garage
resume: false
seed: 42
num_workers: 2
steps: {STEPS}
eval_freq: -1
log_freq: 50
save_checkpoint: true
save_freq: {SAVE_FREQ}
use_policy_training_preset: true
wandb:
  enable: false
'''
YAML = f"{WORK}/smolvla_garage.yaml"
open(YAML, "w").write(yaml_text)
print(yaml_text[:900])

### Checkpoint store helpers (copy the newest checkpoint off the Colab disk)

In [ ]:
def latest_local_ckpt():
    last = Path(LOCAL_OUT) / "checkpoints" / "last"
    return Path(os.path.realpath(last)) if last.exists() else None      # 'last' only moves AFTER a save finished

def _hf_steps():
    try:
        files = api.list_repo_files(HF_MODEL_REPO, repo_type="model")
    except Exception:
        return []
    return sorted({m.group(1) for f in files for m in [re.match(r"checkpoints/(\d+)/", f)] if m})

def sync_to_store():
    src = latest_local_ckpt()
    if src is None:
        return
    if CKPT_STORE == "hf":
        if src.name in _hf_steps():
            return
        api.create_repo(HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(src), repo_id=HF_MODEL_REPO, repo_type="model", path_in_repo=f"checkpoints/{src.name}")
        for old in _hf_steps():                                           # keep only the newest (each is several GB)
            if old != src.name:
                api.delete_folder(path_in_repo=f"checkpoints/{old}", repo_id=HF_MODEL_REPO, repo_type="model")
        print(f"[sync] checkpoint {src.name} saved to {HF_MODEL_REPO}")
    else:
        root = Path(DRIVE_DIR) / "ckpt" / "smolvla_garage" / "checkpoints"
        dst = root / src.name
        if dst.exists():
            return
        root.mkdir(parents=True, exist_ok=True)
        tmp = root / (src.name + ".tmp")
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp, symlinks=False)
        os.rename(tmp, dst)
        for old in root.glob("[0-9]*"):
            if old.name != src.name and not old.name.endswith(".tmp"):
                shutil.rmtree(old, ignore_errors=True)
        print(f"[sync] checkpoint {src.name} saved to Drive")

def restore_from_store():
    ck = Path(LOCAL_OUT) / "checkpoints"
    ck.mkdir(parents=True, exist_ok=True)
    if CKPT_STORE == "hf":
        steps = _hf_steps()
        assert steps, "no checkpoint in the Hugging Face model repo yet"
        name = steps[-1]
        snapshot_download(repo_id=HF_MODEL_REPO, repo_type="model", allow_patterns=f"checkpoints/{name}/*", local_dir=LOCAL_OUT)
    else:
        cks = sorted(p for p in (Path(DRIVE_DIR) / "ckpt" / "smolvla_garage" / "checkpoints").glob("[0-9]*") if not p.name.endswith(".tmp"))
        assert cks, "no checkpoint on Drive yet"
        name = cks[-1].name
        if not (ck / name).exists():
            shutil.copytree(cks[-1], ck / name)
    last = ck / "last"
    if last.is_symlink() or last.exists():
        last.unlink()
    os.symlink(name, last)
    print("restored checkpoint", name)
    return ck / "last" / "pretrained_model" / "train_config.json"

### Train  (output is streamed below; the newest checkpoint is copied to your store every ~90 s)\nThe first start spends several minutes building the dataset cache ('Generating train split').

In [ ]:
stop = threading.Event()
def syncer():
    while not stop.is_set():
        try: sync_to_store()
        except Exception as e: print("[sync] error:", e)
        stop.wait(90)

th = threading.Thread(target=syncer, daemon=True); th.start()
p = subprocess.Popen(f"python train_model.py --config_path {YAML}", shell=True, cwd=WORK,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
stop.set(); th.join()
sync_to_store()
print("training process exited with code", p.returncode)

### [Only after a disconnect / to continue] Resume
Re-run: **login, settings, dataset download, yaml, helpers** (skip the install if the runtime was kept), then run this cell.
`STEPS` in the settings is the final step count, so you can also raise it here to train longer.

In [ ]:
cfg_json = restore_from_store()
stop = threading.Event()
th = threading.Thread(target=syncer, daemon=True); th.start()
p = subprocess.Popen(f"python train_model.py --config_path={cfg_json} --resume=true --steps={STEPS}", shell=True, cwd=WORK,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait(); stop.set(); th.join(); sync_to_store()

### Finished: get the model onto your laptop

In [ ]:
if CKPT_STORE == "hf":
    steps = _hf_steps()
    print("checkpoints in", HF_MODEL_REPO, ":", steps)
    print("""
On your laptop (repo root), with `huggingface-cli login` done once:

    python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='%s', repo_type='model', allow_patterns='checkpoints/%s/pretrained_model/*', local_dir='./ckpt/smolvla_garage')"

That creates ./ckpt/smolvla_garage/checkpoints/%s/pretrained_model  <- set CKPT in 9_multimodal_control.ipynb to this folder.
""" % (HF_MODEL_REPO, steps[-1] if steps else "<step>", steps[-1] if steps else "<step>"))
else:
    root = Path(DRIVE_DIR) / "ckpt" / "smolvla_garage" / "checkpoints"
    for c in sorted(root.glob("[0-9]*")):
        print(c, "->", [x.name for x in (c / "pretrained_model").iterdir()])
    print("Download <that path>/pretrained_model to ./ckpt/smolvla_garage/checkpoints/<step>/pretrained_model on your laptop.")